Isecanje slika

In [ ]:
import os
from PIL import Image

def crop_image(image, n):
    left = 370
    right = 1370
    upper = 540 - n + 1
    lower = 540 + n + 1
    return image.crop((left, upper, right, lower))

def adjust_bounding_box(txt_file):
    with open(txt_file, 'r') as f:
        line = f.readline().strip()
        if line:
            cls, x_center, _, width, _ = map(float, line.split())
            x_center *= 1920
            width *= 1920

            left = x_center - width / 2
            right = x_center + width / 2

            left_new = max(left, 370) - 370
            right_new = min(right, 1370) - 370
        
            if right < 370 or left > 1370:
                return -1, 0, 0
            left_new /= 1000  
            right_new /= 1000
            return int(cls), left_new, right_new
    return -1, 0, 0 


base_input_dir = " " # naziv direktorijuma za ulazne slike i txt fajlove
base_output_dir = " " # naziv direktorijuma za izlazne slike i txt fajlove
n_values = [1, 2, 3]
for n in n_values:
    for split in ['train', 'val']:
        input_images_dir = os.path.join(base_input_dir, "images", split)
        input_labels_dir = os.path.join(base_input_dir, "labels", split)
        output_images_dir = os.path.join(base_output_dir, f"n{n}", "images", split)
        output_labels_dir = os.path.join(base_output_dir, f"n{n}", "labels", split)
        os.makedirs(output_images_dir, exist_ok=True)
        os.makedirs(output_labels_dir, exist_ok=True)

        for img_fname in os.listdir(input_images_dir):
            if img_fname.lower().endswith('.jpg'):
                img_path = os.path.join(input_images_dir, img_fname)
                label_fname = img_fname.replace('.jpg', '.txt')
                label_path = os.path.join(input_labels_dir, label_fname)

                image = Image.open(img_path)
                cropped_image = crop_image(image, n)

                cls, left_new, right_new = adjust_bounding_box(label_path)

                output_img_path = os.path.join(output_images_dir, img_fname)
                cropped_image.save(output_img_path)

                output_label_path = os.path.join(output_labels_dir, label_fname)
                with open(output_label_path, 'w') as f:
                    f.write(f"{cls} {left_new} {right_new}\n")
